In [1]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/model'
os.chdir(workdir)
os.getcwd()

import copy
import importlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
import sklearn

from sklearn.ensemble import RandomForestClassifier as RF
import sklearn.model_selection as skm
from sklearn.metrics import confusion_matrix

In [2]:
pd.set_option('display.max_columns', None)

In [ ]:
import os

job_id = os.environ.get('SLURM_JOB_ID')
if job_id:
    print(f"Running as SLURM job {job_id}")
else:
    print("Not running as a SLURM job")


In [4]:
# The idea:
# 1. Split the dataset into a validation and training dataset
# 2. Train a non-traditional classifier (on the training dataset)
# 3. Estimate label frequency by applying the nontraditional classifier
# on the validation dataset and calculate class prior
# 4. Use class prior for weighing the training dataset and train again
# In Du Plessis paper there is a risk function - see nalysis of Learning from Positive and Unlabeled Data

In [5]:
# df_path = 'saved/gridss_overlaps.tsv'
# callset_overlapped_df = pd.read_csv(df_path, sep='\t', index_col='ID')

df_path = 'saved/gridss.overlapped_bps.csv'
callset_overlapped_df = pd.read_csv(df_path, index_col='INFO_EVENT')
# callset_overlapped_df = callset_overlapped_df.loc[:, ~(callset_overlapped_df.dtypes == 'object')]


In [ ]:
callset_overlapped_df

In [7]:
def get_overlapped_xy(overlapped_df, match_col='is_in_training'):
    overlapped_df_copy = overlapped_df.copy().rename(columns={match_col: 'ref_matched'})
    X = pd.get_dummies(overlapped_df_copy.drop(columns=['ref_matched']))
    # Drop columns with only 1 unique value
    nunique = X.nunique()
    constant_cols = nunique[nunique == 1].index
    X = X.drop(columns=constant_cols)
    y = overlapped_df_copy['ref_matched']
    return X, y

X, y = get_overlapped_xy(callset_overlapped_df)
X_train, X_val, y_train, y_val = skm.train_test_split(X, y, train_size=0.8, random_state=0)

In [8]:
RF_nontrad = RF(max_features=int(np.sqrt(X.shape[1])), random_state=0, n_estimators=500)
RF_nontrad_fit = RF_nontrad.fit(X_train, y_train)
y_nontrad = RF_nontrad_fit.predict(X_val)

label_freq = sum(y_nontrad & y_val) / sum(y_val)
label_prob = sum(y) / len(y)
clpr = label_prob / label_freq

In [9]:
# Weight for True: 2 * clpr / label_prob
# Weight for False: 1 / (1 - label_prob)

In [10]:
# Define class weights based on label probability and class prior ratio
# Weight for negative class (0) is inversely proportional to probability of negative examples
# Weight for positive class (1) accounts for class prior ratio and label probability
weight_dict = {
    0: 1 / (1 - label_prob),  # Weight for negative class
    1: 2 * clpr / label_prob  # Weight for positive class
}

# Initialize Random Forest classifier with:
# - Number of features per split: sqrt(total features) 
# - Fixed random seed for reproducibility
# - Class weights from above
# - 500 trees in the forest
RF_duplessis = RF(
    max_features=int(np.sqrt(X.shape[1])),
    random_state=0,
    class_weight=weight_dict,
    n_estimators=500
)

# Train the classifier on training data
RF_duplessis_fit = RF_duplessis.fit(X_train, y_train)

# Get probability predictions for all examples
# Extract probability of positive class ([:,1])
y_duplessis = RF_duplessis_fit.predict_proba(X)[:,1]

# Save predictions to CSV file
# pd.Series(y_duplessis).to_csv('saved/y_duplessis.csv')
pd.Series(y_duplessis).to_csv('saved/y_duplessis_bps.csv')

In [ ]:
calc_feature_importance = True

if calc_feature_importance:
    import time
    import matplotlib.pyplot as plt
    # Calculate feature importances
    start_time = time.time()
    importances = RF_duplessis_fit.feature_importances_
    std = np.std([
        tree.feature_importances_ 
        for tree in RF_duplessis_fit.estimators_
    ], axis=0)
    elapsed_time = time.time() - start_time

    print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")

    # Create visualization
    forest_importances = pd.Series(importances, index=X.columns)

    fig, ax = plt.subplots()
    forest_importances.plot.bar(yerr=std, ax=ax)
    ax.set_title("Feature Importances using MDI")
    ax.set_ylabel("Mean Decrease in Impurity")
    fig.tight_layout()

In [ ]:
from sklearn.inspection import permutation_importance
import time

start_time = time.time()
result = permutation_importance(
    RF_duplessis_fit, X_val, y_val, n_repeats=10, random_state=42, n_jobs=2
)
elapsed_time = time.time() - start_time
print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")

forest_importances = pd.Series(result.importances_mean, index=X.columns)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
forest_importances.plot(yerr=result.importances_std, ax=ax, kind='bar')
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()
plt.show()

In [ ]:

fig, ax = plt.subplots()
forest_importances.plot(xerr=result.importances_std, ax=ax, kind='barh')
ax.set_title("Mean accuracy decrease")
ax.set_ylabel("Feature importances using permutation on full model")

fig.tight_layout()
fig.set_size_inches((5,20))
plt.show()
# plt.show()
plt.savefig('figs/featureimportance_perm.enoto.png')
# Save figure with estimated size to ensure text is readable


In [33]:
# I am not sure it is possible to do a custom loss function in sklearn - I DO NOT NEED TO, I CAN JUST USE WEIGHTS FOR DUPLESSIS CORRECTION
# So I implement Elkan and Noto sample weighting scheme
# Where unlabeled samples are split into 2:
# - positive part with weight (1 - label_freq) / label_freq * y_nontrad / (1 - y_nontrad)
# - negative part with weight 1 - positive_weight

In [18]:
y_proba_nontrad = RF_nontrad.predict_proba(X_train)
pos_weights = y_proba_nontrad[:, 1]
pos_weights[y_train] = 1

In [20]:
# We double the unlabeled part of X and y and give negative weights to it
X_train_neg = X_train[y_train == False]
y_train_neg = y_train[y_train == False]
neg_weights = y_proba_nontrad[y_train == False, 0]
X_doubled = pd.concat([X_train, X_train_neg])
y_doubled = np.concatenate([y_train, y_train_neg])
weights_doubled = np.concatenate([pos_weights, neg_weights])
RF_enoto = RF(
    max_features=int(np.sqrt(X.shape[1])), random_state=0, n_estimators=500
)
RF_enoto_fit = RF_enoto.fit(X_doubled, y_doubled, sample_weight=weights_doubled)

In [ ]:
X_doubled

In [21]:
y_enoto = RF_enoto_fit.predict_proba(X)[:,1]

In [24]:
pd.DataFrame({
    'ID': callset_overlapped_df.index,
    'y_duplessis': y_duplessis,
    'y_enoto': y_enoto
}).to_csv('saved/y_duplessis_enoto_bps.csv', index=False)

In [22]:
# pd.Series(y_enoto).to_csv('saved/y_enoto.csv')
pd.Series(y_enoto).to_csv('saved/y_enoto_bps.csv')


In [ ]:
from sklearn.inspection import permutation_importance

start_time = time.time()
result = permutation_importance(
    RF_enoto_fit, X_val, y_val, n_repeats=10, random_state=42, n_jobs=2
)
elapsed_time = time.time() - start_time
print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")

forest_importances = pd.Series(result.importances_mean, index=X.columns)

In [ ]:

fig, ax = plt.subplots()
forest_importances.plot(xerr=result.importances_std, ax=ax, kind='barh')
ax.set_title("Mean accuracy decrease")
ax.set_ylabel("Feature importances using permutation on full model")

fig.tight_layout()
fig.set_size_inches((5,10))
plt.show()
# plt.show()
# plt.savefig('figs/featureimportance_perm.enoto.png')

In [ ]:
disp = CalibrationDisplay(prob_true, prob_pred, y_prob)
disp.plot()